# Save Cell Type Metadata

**Run this notebook ONCE before `cell_type_importance.ipynb`.**

### What it does
Saves a per-donor `{donor_id}_celltypes.npy` array alongside each patient's data.  
It runs the **exact same DataModule preprocessing as `train.py`** (HVG selection, normalisation, subsampling) so the cell ordering in the saved arrays is guaranteed to match what `SingleCellMetricModel` sees at inference time.

### Output
```
data/cell_type_metadata/
    donor001_celltypes.npy    (n_cells,)  str
    donor002_celltypes.npy
    ...
```

### Prerequisites
- The original `.h5ad` file used for training
- `src/data_loader.py` (DataModule) on the Python path

## 1 · Imports

In [ ]:
import os
import sys
import numpy as np
import scanpy as sc
from pathlib import Path

# ── Make sure the repo root is on the path so src.data_loader resolves ────────
REPO_ROOT = Path(".").resolve()   # change if running from a subdirectory
sys.path.insert(0, str(REPO_ROOT))

from src.data_loader import DataModule

print("Imports OK ✓")

## 2 · Configuration

> **⚠️ Edit this cell to match your `train.py` run exactly.**  
> All values must be identical to those used during training so that  
> preprocessing produces the same cell ordering.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
H5AD_PATH   = "data/SCP1884_subset_50donors.h5ad"
MODEL_DIR   = "scgpt/continual_pretrained"   # must contain vocab.json
OUTPUT_DIR  = "data/cell_type_metadata/"     # where to save *_celltypes.npy

# ── DataModule params — copy from train.py DEFAULTS ──────────────────────────
GENE_COL              = "index"
DONOR_COL             = "donor_id"
LABEL_COL             = "disease__ontology_label"
DISEASE_LABEL         = "Crohn's disease"
MAX_CELLS_PER_PATIENT = 5000
N_HVG                 = 5000
SEED                  = 42

# ── Cell type column ──────────────────────────────────────────────────────────
# Check adata.obs.columns — common names: "cell_type", "cell_type_fine",
# "leiden", "celltype_l2", "Celltype", etc.
CELL_TYPE_COL = "cell_type"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory : {OUTPUT_DIR}")

## 3 · Load AnnData and inspect

In [ ]:
print("Loading AnnData ...")
adata = sc.read_h5ad(H5AD_PATH)

print(f"  Total cells   : {adata.shape[0]:,}")
print(f"  Total genes   : {adata.shape[1]:,}")
print(f"  obs columns   : {adata.obs.columns.tolist()}")
print(f"  Unique donors : {adata.obs[DONOR_COL].nunique()}")
print(f"  Cell types    : {sorted(adata.obs[CELL_TYPE_COL].unique().tolist())}")

## 4 · Run DataModule preprocessing

This mirrors `train.py` exactly: HVG selection, normalisation, and vocabulary filtering.  
Cell type labels are read from `adata.obs` **after** this step so they correspond to the cells the model actually sees.

In [ ]:
vocab_path = str(Path(MODEL_DIR) / "vocab.json")

print("Building DataModule and running preprocess_for_scgpt ...")
dm = DataModule(
    adata                 = adata,
    gene_vocab_file       = vocab_path,
    gene_col              = GENE_COL,
    patient_col           = DONOR_COL,
    bag_col               = None,
    label_col             = LABEL_COL,
    disease_label         = DISEASE_LABEL,
    max_cells_per_patient = MAX_CELLS_PER_PATIENT,
    batch_size            = 1,
)
dm.preprocess_for_scgpt(n_hvg=N_HVG)

# Access the preprocessed AnnData.
# If your DataModule stores it under a different attribute (e.g. dm.adata_,
# dm.processed_adata), update the line below.
preprocessed_adata = dm.adata

print(f"\nPreprocessed shape : {preprocessed_adata.shape}")
print(f"HVGs kept          : {preprocessed_adata.shape[1]}")

# Guard: make sure the cell type column is still present
assert CELL_TYPE_COL in preprocessed_adata.obs.columns, (
    f"Column '{CELL_TYPE_COL}' not found after preprocessing.\n"
    f"Available columns: {preprocessed_adata.obs.columns.tolist()}"
)
print(f"Cell type column '{CELL_TYPE_COL}' present ✓")

## 5 · Helper — filesystem-safe donor ID

In [ ]:
def safe_donor_id(donor_id: str) -> str:
    """Mirrors DataModule's safe_did logic: replace / with _."""
    return donor_id.replace("/", "_")

## 6 · Save per-donor cell type arrays

For each donor:
1. Slice cells from the preprocessed AnnData
2. Subsample to `MAX_CELLS_PER_PATIENT` if needed (same logic as DataModule)
3. Save as `{safe_donor_id}_celltypes.npy`

> **Subsampling note:** Two scenarios are shown below.  
> Check your `DataModule` source to confirm which one it uses.

In [ ]:
donors = sorted(preprocessed_adata.obs[DONOR_COL].unique())
print(f"Saving cell type arrays for {len(donors)} donors → {OUTPUT_DIR}\n")

saved, errors = 0, []

for donor_id in donors:
    mask        = preprocessed_adata.obs[DONOR_COL] == donor_id
    donor_cells = preprocessed_adata[mask]

    cell_types = donor_cells.obs[CELL_TYPE_COL].values.astype(str)
    n_total    = len(cell_types)

    # ── Subsample to match DataModule ────────────────────────────────────────
    # Scenario A — DataModule takes the FIRST N cells (deterministic):
    #   cell_types = cell_types[:MAX_CELLS_PER_PATIENT]
    #
    # Scenario B — DataModule samples randomly with a fixed seed (default below):
    if n_total > MAX_CELLS_PER_PATIENT:
        rng = np.random.default_rng(SEED)
        idx = rng.choice(n_total, MAX_CELLS_PER_PATIENT, replace=False)
        idx.sort()           # preserve original cell order
        cell_types = cell_types[idx]

    safe_did  = safe_donor_id(donor_id)
    save_path = os.path.join(OUTPUT_DIR, f"{safe_did}_celltypes.npy")
    np.save(save_path, cell_types)
    saved += 1

    print(f"  {safe_did:<40} {len(cell_types):>5}/{n_total:<5} cells  "
          f"types={sorted(set(cell_types))[:3]}...")

print(f"\nDone. Saved {saved} cell type files.")
if errors:
    print(f"Errors ({len(errors)}): {errors}")

## 7 · Verification

Spot-check the first donor to confirm shape and content look correct.

In [ ]:
sample_donor = safe_donor_id(donors[0])
ct_check = np.load(
    os.path.join(OUTPUT_DIR, f"{sample_donor}_celltypes.npy"),
    allow_pickle=True
)

print(f"Verification — {sample_donor}:")
print(f"  Cells saved   : {ct_check.shape[0]}")
print(f"  Unique types  : {np.unique(ct_check).tolist()}")
print()
print("✓ Cell type metadata ready — run cell_type_importance.ipynb next.")